In [56]:
import pandas as pd
import numpy as np
import re
from functools import reduce
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# --- 1. Read and merge MASTER.xlsx sheets ---

all_sheets = pd.read_excel("MASTER.xlsx", sheet_name=None)
dfs = []
gp_df = None

for name, df in all_sheets.items():

    df = df.rename(columns=lambda c: c.upper().strip())

    if "PLAYER" not in df.columns or "GP" not in df.columns:
        raise KeyError(f"Sheet '{name}' missing PLAYER or GP column.")
    
    if name.strip().upper() == "GENERAL - DEFENSE":
        gp_df = df[["PLAYER", "GP"]].drop_duplicates()

    # Drop GP from this df
    df = df.drop(columns="GP")
    
    # Rename other columns with sheet name suffix
    df = df.rename(columns={col: f"{col}_{name.strip().upper()}" for col in df.columns if col != "PLAYER"})
    
    dfs.append(df)

if gp_df is None:
    raise KeyError("Could not find sheet 'General - Defense' to extract GP.")

merged_df = reduce(lambda L, R: pd.merge(L, R, on="PLAYER", how="outer"), dfs)
merged_df = merged_df.merge(gp_df, on="PLAYER", how="left")

In [58]:
# --- 2. Merge Def Impact Score from Rankings.xlsx ---

scores = pd.read_excel("Rankings.xlsx", sheet_name="Def Impact Score")
scores = scores.rename(columns=lambda c: c.upper().strip())[["PLAYER", "SCORE"]]

def clean_name(name: str) -> str:
    name = name.upper().strip()
    name = re.sub(r"[.\-]", "", name)
    return re.sub(r"[^A-Z0-9 ]", ".", name)

merged_df["PLAYER"] = merged_df["PLAYER"].apply(clean_name)
scores["PLAYER"] = scores["PLAYER"].apply(clean_name)

merged_df = (
    merged_df.merge(scores, on="PLAYER", how="left")
             .dropna(subset=["SCORE"])
)

merged_df = merged_df.sort_values(by="SCORE", ascending=False).reset_index(drop=True)

# Print unmatched players
unmatched_players = set(scores["PLAYER"]) - set(merged_df["PLAYER"])
print("Players in Rankings.xlsx not matched in merged_df:")
print(sorted(unmatched_players))

merged_df

Players in Rankings.xlsx not matched in merged_df:
['CJ.MCCOLLUM', 'DORIAN FINNEY.SMITH', 'GARY PAYTON.II', 'HERB JONES', 'JAREN JACKSON.JR', 'JIMMY BUTLER', 'KELLY OUBRE.JR', 'KENTAVIOUS CALDWELL.POPE', 'ROBERT WILLIAMS.III']


,PLAYER,TEAM_GENERAL - DEFENSE,AGE_GENERAL - DEFENSE,W_GENERAL - DEFENSE,L_GENERAL - DEFENSE,MIN_GENERAL - DEFENSE,DEF RTG_GENERAL - DEFENSE,DREB_GENERAL - DEFENSE,DREB%_GENERAL - DEFENSE,%DREB_GENERAL - DEFENSE,...,OFF BOX OUTS_BOX OUTS,DEF BOX OUTS_BOX OUTS,TEAM REB ON BOX OUTS_BOX OUTS,PLAYER REB ON BOX OUTS_BOX OUTS,% BOX OUTS OFF_BOX OUTS,% BOX OUTS DEF_BOX OUTS,% TEAM REB WHEN BOX OUT_BOX OUTS,% PLAYER REB WHEN BOX OUT_BOX OUTS,GP,SCORE
0,RUDY GOBERT,MIN,32,53,23,34.1,106.6,9.2,25.6,36.8,...,0.9,1.9,2.8,1.9,33.0,67.0,99.5,68.4,76,561.0
1,VICTOR WEMBANYAMA,SAS,20,19,52,29.7,111.2,8.4,27.3,38.9,...,0.3,0.6,0.8,0.5,32.8,67.2,100.0,57.6,71,560.0
2,BAM ADEBAYO,MIA,26,40,31,34.0,109.3,8.1,24.2,34.9,...,0.3,2.8,2.9,1.4,10.0,90.0,97.6,48.8,71,559.0
3,ANTHONY DAVIS,LAL,31,45,31,35.5,114.0,9.5,25.2,36.6,...,0.6,1.8,2.5,1.7,25.9,74.1,98.4,68.8,76,557.0
4,ALEX CARUSO,CHI,30,34,37,28.7,110.6,3.0,10.3,15.4,...,0.0,0.4,0.4,0.1,6.9,93.1,96.6,31.0,71,556.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,JAVONTE GREEN,CHI,30,5,4,25.6,112.8,5.2,21.9,29.9,...,0.4,1.1,1.6,0.9,28.6,71.4,100.0,57.1,9,473.0
79,TORREY CRAIG,CHI,33,22,31,19.8,118.4,2.8,14.8,21.6,...,0.1,0.4,0.4,0.3,13.6,86.4,100.0,63.6,53,471.0
80,HAMIDOU DIALLO,WAS,25,1,1,2.4,50.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,470.0
81,THEO MALEDON,PHX,23,5,12,12.5,121.6,1.2,9.3,14.1,...,0.0,0.1,0.1,0.0,0.0,100.0,100.0,0.0,17,469.0


In [59]:
# Load the defender scores CSV
defender_df = pd.read_csv("defender_scores_2024.csv")


defender_df["PLAYER"] = (defender_df["first_name"] + " " + defender_df["family_name"]).str.upper().str.strip()

defender_df["PLAYER"] = defender_df["PLAYER"].apply(clean_name)

defender_df = defender_df.drop_duplicates(subset="PLAYER")


# Uppercase PLAYER in merged_df for consistency
merged_df["PLAYER"] = merged_df["PLAYER"].str.upper().str.strip()

# Now merge
merged_df = merged_df.merge(defender_df[["PLAYER", "avg_scoring_probability", "possessions"]], on="PLAYER", how="left")

# drop duplicates
merged_df = merged_df.drop_duplicates(subset="PLAYER")




unmatched = merged_df[merged_df["avg_scoring_probability"].isna()]
print("Unmatched rows:", len(unmatched))

Unmatched rows: 47


In [60]:
merged_df

,PLAYER,TEAM_GENERAL - DEFENSE,AGE_GENERAL - DEFENSE,W_GENERAL - DEFENSE,L_GENERAL - DEFENSE,MIN_GENERAL - DEFENSE,DEF RTG_GENERAL - DEFENSE,DREB_GENERAL - DEFENSE,DREB%_GENERAL - DEFENSE,%DREB_GENERAL - DEFENSE,...,TEAM REB ON BOX OUTS_BOX OUTS,PLAYER REB ON BOX OUTS_BOX OUTS,% BOX OUTS OFF_BOX OUTS,% BOX OUTS DEF_BOX OUTS,% TEAM REB WHEN BOX OUT_BOX OUTS,% PLAYER REB WHEN BOX OUT_BOX OUTS,GP,SCORE,avg_scoring_probability,possessions
0,RUDY GOBERT,MIN,32,53,23,34.1,106.6,9.2,25.6,36.8,...,2.8,1.9,33.0,67.0,99.5,68.4,76,561.0,0.483147,498.0
1,VICTOR WEMBANYAMA,SAS,20,19,52,29.7,111.2,8.4,27.3,38.9,...,0.8,0.5,32.8,67.2,100.0,57.6,71,560.0,0.293895,334.0
2,BAM ADEBAYO,MIA,26,40,31,34.0,109.3,8.1,24.2,34.9,...,2.9,1.4,10.0,90.0,97.6,48.8,71,559.0,0.396964,517.0
3,ANTHONY DAVIS,LAL,31,45,31,35.5,114.0,9.5,25.2,36.6,...,2.5,1.7,25.9,74.1,98.4,68.8,76,557.0,0.332713,365.0
4,ALEX CARUSO,CHI,30,34,37,28.7,110.6,3.0,10.3,15.4,...,0.4,0.1,6.9,93.1,96.6,31.0,71,556.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,JAVONTE GREEN,CHI,30,5,4,25.6,112.8,5.2,21.9,29.9,...,1.6,0.9,28.6,71.4,100.0,57.1,9,473.0,NaN,NaN
79,TORREY CRAIG,CHI,33,22,31,19.8,118.4,2.8,14.8,21.6,...,0.4,0.3,13.6,86.4,100.0,63.6,53,471.0,NaN,NaN
80,HAMIDOU DIALLO,WAS,25,1,1,2.4,50.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,2,470.0,NaN,NaN
81,THEO MALEDON,PHX,23,5,12,12.5,121.6,1.2,9.3,14.1,...,0.1,0.0,0.0,100.0,100.0,0.0,17,469.0,NaN,NaN


In [61]:
# --- 3. Prepare features and target (model training) ---

numeric_cols = merged_df.select_dtypes(include="number").columns.drop("SCORE")
X = merged_df[numeric_cols].values
y = merged_df["SCORE"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [62]:
# --- 4. Fit RandomForest with LOOCV evaluation ---

rf = RandomForestRegressor(n_estimators=100, random_state=42)

loo = LeaveOneOut()
y_pred = cross_val_predict(rf, X_scaled, y, cv=loo)

r2 = r2_score(y, y_pred)
rmse = mean_squared_error(y, y_pred) ** 0.5
print(f"RandomForest LOOCV R²:  {r2:.3f}")
print(f"RandomForest LOOCV RMSE: {rmse:.3f}")

# Fit final model on full data
rf.fit(X_scaled, y)
importances = pd.Series(rf.feature_importances_, index=numeric_cols)
importances = importances.sort_values(ascending=False)

print("\nRandomForest Feature Importances:")
print(importances)

RandomForest LOOCV R²:  0.345
RandomForest LOOCV RMSE: 22.063

RandomForest Feature Importances:
DEF WS_GENERAL - DEFENSE                0.335267
MIN_GENERAL - DEFENSE                   0.042073
MIN_BOX OUTS                            0.041113
% TEAM REB WHEN BOX OUT_BOX OUTS        0.032250
BLK_GENERAL - DEFENSE                   0.031312
MIN_HUSTLE                              0.031064
OPP PTS 2ND CHANCE_GENERAL - DEFENSE    0.025329
DFGM_DEFENSE DASHBOARD                  0.023657
DEF RTG_GENERAL - DEFENSE               0.023299
STL%_GENERAL - DEFENSE                  0.022294
CONTESTED 3 PT SHOTS_HUSTLE             0.022023
%BLK_GENERAL - DEFENSE                  0.021000
DFG%_DEFENSE DASHBOARD                  0.017223
% BOX OUTS OFF_BOX OUTS                 0.017183
DFGA_DEFENSE DASHBOARD                  0.017067
% PLAYER REB WHEN BOX OUT_BOX OUTS      0.016566
CONTESTED SHOTS_HUSTLE                  0.014464
DEFLECTIONS_HUSTLE                      0.013870
OPP PTS FB_GENERAL - 

In [63]:
# --- 5. Add new impact score based on dynamic feature importances ---

# Work on a new copy of merged_df to avoid scaled columns
output_df = merged_df.copy()

# Apply weights only to columns that exist
for feature, weight in importances.items():
    if feature in output_df.columns:
        output_df[feature + "_WEIGHTED"] = output_df[feature] * weight

# Compute impact score
weighted_cols = [col for col in output_df.columns if col.endswith("_WEIGHTED")]
output_df["IMPACT SCORE"] = output_df[weighted_cols].sum(axis=1)

In [64]:
# --- 6. Filter by GP after metric creation ---
output_df = output_df[output_df["GP"] >= 65]

In [65]:
# --- 7. Sort and save to CSV (with original numeric columns unscaled) ---
output_df = output_df.sort_values("IMPACT SCORE", ascending=False)
output_df.to_csv("OUTPUT.csv", index=False)